# The Amazon's Rondônia fishbone, 2001 → 2025

Along BR-364 in Rondônia, Brazil, three decades of settlement carved the Amazon into one of the most recognisable deforestation patterns from orbit — the **fishbone**: a highway trunk with perpendicular farm roads branching off it, each one widening its own strip of cleared land into the forest. This notebook animates 25 years of it, pixel by pixel, from a single Earth Engine request.

That request goes through earthlens' **`gee`** backend — a unified interface over **1,104 curated Google Earth Engine datasets across 10 categories** (land-cover-change, optical-multispectral, hydrology-water, and seven more), each with typed band/cadence/license metadata, behind the same `EarthLens(data_source="gee", ...)` call shape as the other 60+ earthlens backends. Normally, using Earth Engine means learning its own JS/Python API, its auth flow, and hunting asset ids in the catalog UI — this notebook is one request.

In [ ]:
import base64
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import numpy as np
from cleopatra.glyphs.base.animation import gif_from_video
from cleopatra.styling.styles import apply_blank_canvas
from cleopatra.styling.watermark import stamp_mark
from IPython.display import HTML
from matplotlib.colors import ListedColormap
from pyramids.dataset import Dataset, GeoReference
from pyramids.dataset.collection import DatasetCollection
from pyramids.plot import FrameLabel

from earthlens.core import EarthLens

SERVICE_ACCOUNT = os.environ["GEE_SERVICE_ACCOUNT"]
SERVICE_KEY = os.environ["GEE_SERVICE_KEY"]

OUT = Path("out") / "amazon_deforestation"
OUT.mkdir(parents=True, exist_ok=True)
FRAMES = OUT / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

# The Rondonia fishbone along BR-364, near Ariquemes / Machadinho d'Oeste.
RONDONIA = {"lat_lim": [-10.3, -9.3], "lon_lim": [-63.5, -62.3]}

# The overlay lockup carries its own navy scrim, built for watermarking over
# arbitrary/dark imagery -- see docs/_images/branding/earthlens-brand-kit/BRAND-GUIDE.md.
LOGO = Path(
    "../../_images/branding/earthlens-brand-kit/logo/earthlens-lockup-stacked-overlay.png"
)

# LinkedIn's recommended single-image post size (1200x627, 1.91:1) at dpi=150.
SOCIAL_FIGSIZE = (8.0, 4.18)


def stamp_watermark(
    fig, *, text="earthlens", angle=30, text_alpha=0.65, credit=None, credit_alpha=1.0
):
    """A soft, translucent diagonal 'earthlens' text watermark, plus a bottom credit line.

    Plain semi-transparent white text with no outline -- an outline reads as a
    solid caption rather than a watermark. The credit line keeps its stroke
    since it has to stay legible as a small line of text against any frame
    content. The corner brand mark from `stamp_mark` covers the logo itself.
    """
    fig_w_in, _ = fig.get_size_inches()
    fig.text(
        0.5,
        0.5,
        text,
        rotation=angle,
        ha="center",
        va="center",
        fontsize=fig_w_in * 6,
        fontweight="bold",
        color="white",
        alpha=text_alpha,
        zorder=1_000_000,
    )
    if credit:
        fig.text(
            0.5,
            0.014,
            credit,
            ha="center",
            va="bottom",
            color="white",
            fontsize=7.5,
            alpha=credit_alpha,
            zorder=1_000_002,
            path_effects=[
                patheffects.withStroke(linewidth=2, foreground="black", alpha=0.9)
            ],
        )

## 1 · Fetch the Hansen Global Forest Change record, once

[Hansen Global Forest Change](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2025-v1.13/download.html) (UMD/Google/USGS/NASA) is the record behind [Global Forest Watch](https://www.globalforestwatch.org/) -- a single 30 m image, not a time series, with `treecover2000` (percent canopy cover in the baseline year) and `lossyear` (0 = never lost, 1-25 = the calendar year forest was lost, 2001-2025). One request for both bands is everything the whole animation below is built from.

In [ ]:
raw_tif = OUT / "raw" / "hansen_treecover2000_lossyear.tif"
if not raw_tif.exists():
    job = EarthLens(
        data_source="gee",
        variables={
            "UMD/hansen/global_forest_change_2025_v1_13": ["treecover2000", "lossyear"]
        },
        start="2000-01-01",
        end="2025-12-31",
        temporal_resolution="raw",
        path=raw_tif.parent,
        scale=100.0,
        **RONDONIA,
    )
    job.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
    fetched = job.download()
    fetched[0].rename(raw_tif)

hansen = Dataset.read_file(str(raw_tif))
hansen.rows, hansen.columns

## 2 · One frame per year, all derived from that single fetch

`lossyear` already encodes *when* each pixel was cleared, so every year's frame is a threshold on the same array -- no second request. A pixel is: **standing forest** if `treecover2000 >= 30%` and it either never lost cover or won't be lost until after this frame's year; **deforested** if it's a forest pixel whose loss year has already passed; otherwise **non-forest** (it was never forest in 2000 -- water, bare ground, existing clearings, settlements).

In [ ]:
arr = hansen.read_array()
treecover2000, lossyear = arr[0], arr[1]
forest_mask = treecover2000 >= 30
geo_ref = GeoReference(geo=hansen.geotransform, epsg=hansen.epsg)

years = list(range(2001, 2026))
frame_paths = []
for year in years:
    frame_path = FRAMES / f"{year}.tif"
    if not frame_path.exists():
        year_offset = year - 2000
        category = np.zeros(treecover2000.shape, dtype=np.uint8)
        category[forest_mask] = 1
        deforested = forest_mask & (lossyear > 0) & (lossyear <= year_offset)
        category[deforested] = 2
        Dataset.from_array(category, geo_ref=geo_ref, no_data_value=None).to_file(
            str(frame_path)
        )
    frame_paths.append(frame_path)

forest_2000 = int(forest_mask.sum())
lost_by_2025 = int((forest_mask & (lossyear > 0)).sum())
pct_lost = 100 * lost_by_2025 / forest_2000
len(frame_paths), f"{pct_lost:.1f}% of this box's year-2000 forest lost by 2025"

## 3 · Animate, and export both `.gif` and `.mp4`

Rendered at exactly `SOCIAL_FIGSIZE` (1200x627 px, LinkedIn's recommended single-image post size) -- `figsize=` passed to `.plot()` is only a hint cleopatra overrides from the data's own aspect ratio, so the exact size is re-forced via `set_size_inches` after `add_reference_map`. It's rendered once to `.mp4` (ready for a native LinkedIn video post) and the `.gif` embedded below is derived from that file via cleopatra's `gif_from_video`, rather than re-rendering the whole animation a second time. The full-range `scale` filter and `color_range` tag avoid the classic FFmpeg limited-range washout -- without them the exported video loses real contrast (confirmed by inspecting the raw encoded bytes; reported upstream as [serapeum-org/cleopatra#344](https://github.com/serapeum-org/cleopatra/issues/344)).

In [ ]:
land_cover_cmap = ListedColormap(["#C9B896", "#1B7A3D", "#D62728"])
west, north = hansen.geotransform[0], hansen.geotransform[3]
east = west + hansen.geotransform[1] * hansen.columns
south = north + hansen.geotransform[5] * hansen.rows

glyph = DatasetCollection.from_files([str(p) for p in frame_paths]).plot(
    cmap=land_cover_cmap,
    vmin=0,
    vmax=2,
    figsize=SOCIAL_FIGSIZE,
    animation_axis_values=[str(y) for y in years],
    frame_label=FrameLabel(color="white", size=14),
    colorbar=False,
)
apply_blank_canvas(glyph.ax, facecolor="black")
glyph.add_reference_map(style="dark", extent=[west, south, east, north])
# figsize= passed to .plot() is only a hint cleopatra overrides from the data's own
# aspect ratio, so re-force the exact size for a consistent, LinkedIn-ready canvas.
glyph.fig.set_size_inches(*SOCIAL_FIGSIZE)
glyph.fig.set_dpi(150)

legend_handles = [
    mpatches.Patch(color="#1B7A3D", label="Standing forest"),
    mpatches.Patch(color="#D62728", label="Deforested since 2000"),
    mpatches.Patch(color="#C9B896", label="Non-forest"),
]
glyph.fig.legend(
    handles=legend_handles,
    loc="upper right",
    bbox_to_anchor=(0.98, 0.95),
    facecolor="black",
    edgecolor="none",
    labelcolor="white",
    fontsize=8,
    framealpha=0.6,
)

# Both bake their position from the figure's CURRENT size, so they must come
# after set_dpi/set_size_inches -- and before save_animation, which reads the
# figure as its final frame.
stamp_mark(glyph.fig, str(LOGO), frac=0.18, corner="lower left")
stamp_watermark(glyph.fig, credit="github.com/serapeum-org/earthlens")

mp4_path = OUT / "amazon_deforestation.mp4"
gif_path = OUT / "amazon_deforestation.gif"
glyph.save_animation(
    str(mp4_path),
    fps=5,
    pix_fmt="yuv444p",
    crf=18,
    extra_args=["-vf", "scale=out_range=full", "-color_range", "pc"],
)
gif_from_video(str(mp4_path), str(gif_path), fps=5)
plt.close("all")

encoded = base64.b64encode(gif_path.read_bytes()).decode()
HTML(
    f'<img src="data:image/gif;base64,{encoded}" '
    'alt="Rondonia fishbone deforestation, 2001-2025" />'
)

## Recap

One Earth Engine request through earthlens' `gee` backend -- `treecover2000` and `lossyear` from Hansen Global Forest Change -- reduced to 25 yearly frames entirely client-side, animated through pyramids/cleopatra, exported as both a LinkedIn-ready GIF and MP4. Across this box of Rondônia's BR-364 fishbone, roughly **47% of the standing forest recorded in 2000 was gone by 2025**. Every pixel here comes from the same public, no-credential-required-beyond-a-GEE-service-account catalog row -- swap `RONDONIA` for another `lat_lim`/`lon_lim` box (the Congo Basin, Sumatra, the Chaco) and the rest of the notebook runs unchanged.